In [ ]:
import json
import ollama
from dataclasses import dataclass 


@dataclass
class Config:
    llm_model: str = "llama3.1:8b"
    embed_model: str = "nomic-embed-text"  
    ollama_host: str = "http://localhost:11434"
    temperature: float = 0.0


config = Config()
client = ollama.Client(host=config.ollama_host)


def llm_json(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat(
        model=config.llm_model, messages=messages,
        format="json", options={"temperature": config.temperature},
    )
    raw = resp["message"]["content"]
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start, end = raw.find("{"), raw.rfind("}")
        return json.loads(raw[start: end + 1])


def llm_text(prompt, system=None):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    resp = client.chat(
        model=config.llm_model, messages=messages,
        options={"temperature": config.temperature},
    )
    return resp["message"]["content"].strip()


def llm_embed(text):

    try:
        resp = client.embed(model=config.embed_model, input=text)
        return resp["embeddings"][0]
    except Exception as e:
        print(f"[llm_embed warning] {e}")
        return None

try:
    client.list()
    print(f"Ollama conectado en {config.ollama_host}")
    print(f"  LLM   : {config.llm_model}")
    print(f"  EMBED : {config.embed_model}")
    test_emb = llm_embed("hola mundo")
    if test_emb is None:
        print("  Embeddings NO disponibles. Ejecuta: ollama pull nomic-embed-text")
    else:
        print(f"  Embeddings OK (dim={len(test_emb)})")
except Exception as e:
    print("Ollama no responde:", e)


Ollama conectado en http://localhost:11434
  LLM   : llama3.1:8b
  EMBED : nomic-embed-text
  Embeddings OK (dim=768)


# Fase 1 — Extracción de información

Por cada turno, el LLM extrae:
- **entidades nombradas** (personas, lugares, organizaciones, ...)
- **resumen factual** (preserva nombres, fechas, números)
- **tópicos** (2-5 keywords)


In [ ]:
TITLE_PREFIXES = {
    "dr", "doctor", "mr", "mister", "mrs", "ms", "miss",
    "prof", "professor", "chef", "sir", "lady", "captain", "officer",
}


def strip_title_prefix(name):
    parts = name.split("_")
    if len(parts) > 1 and parts[0] in TITLE_PREFIXES:
        return "_".join(parts[1:])
    return name


ENTITY_EXTRACTOR_PROMPT = """You are an entity extractor for a long-term memory system.
The speaker is: {speaker}

Any first-person pronoun (I, me, my, mine, we, our) refers to {speaker}.
Always include {speaker} as a "person" entity when first-person pronouns appear.

Extract ONLY entities that have an EXPLICIT, SPECIFIC NAME in the text.
For each, output:
- "name": canonical lowercase identifier with underscores (no spaces, no articles).
- "type": one of [person, location, organization, object, concept, event, date, attribute].

CANONICAL NAMING RULES:
- ALWAYS strip titles ("Dr.", "Mr.", "Mrs.", "Ms.", "Prof.", "chef", ...).
  * "Dr. Maria Garcia" -> "maria_garcia"   (NOT "dr_maria_garcia")
  * "chef Marco"        -> "marco"         (NOT "chef_marco")
- Use only the proper-noun part of the name.
- Place/brand examples:
  * "the Eiffel Tower"   -> "eiffel_tower"
  * "Microsoft Office"   -> "microsoft_office"
  * "my brother John"    -> "john"

WHAT TO EXTRACT:
- Specific, NAMED people, places, organizations, products, named events, dates.

WHAT IS *NOT* AN ENTITY -- DO NOT EXTRACT:
- Unnamed references ("my brother", "my house", "my job", "where I live") -> OMIT.
- NEVER invent placeholder names like "speaker_brother", "user_home", "user_X".
- Generic concepts ("food", "happiness", "things", "home", "work", "park").
- Question-frame concepts (don't invent entities for what is being asked).
- Filler words.

If only unnamed references appear, return only the speaker:
{{"entities": [{{"name": "{speaker}", "type": "person"}}]}}

OUTPUT — return ONLY this JSON:
{{"entities": [{{"name": "<name>", "type": "<type>"}}, ...]}}

CONVERSATION:
---
{conversation}
---
"""


def extract_entities(text, speaker):
    prompt = ENTITY_EXTRACTOR_PROMPT.format(conversation=text, speaker=speaker)
    result = llm_json(prompt)
    cleaned = []
    for e in result.get("entities", []):
        name = e.get("name", "").strip().lower()
        if not name:
            continue
        if name != speaker and (name.startswith(f"{speaker}_")
                                or name.startswith("speaker_")):
            continue
        name = strip_title_prefix(name)
        if not name:
            continue
        cleaned.append({"name": name, "type": e.get("type", "concept")})
    return cleaned


In [ ]:
# Extractor de resumen y tópicos
SUMMARY_PROMPT = """You are a memory system processing a conversation turn.

Generate a 1-2 sentence summary (third-person, max ~30 words) that PRESERVES all 
specific factual values mentioned: numbers, money, times, dates, names, brands, 
durations, model numbers, identifying details.

A good summary is a factual record, not a vague description.

Examples:

  Original: "I just got a 2020 Honda Civic for $18,500 last weekend."
  GOOD: "The user bought a 2020 Honda Civic for $18,500 last weekend."
  BAD:  "The user mentioned a car purchase."

  Original: "On Mondays I have therapy at 4pm with Dr. Lopez."
  GOOD: "The user has therapy with Dr. Lopez at 4pm on Mondays."
  BAD:  "The user described their therapy schedule."

Also extract 2-5 short lowercase topics.

Speaker: {speaker}

OUTPUT FORMAT — return ONLY this JSON:
{{"summary": "<factual summary preserving specific values>", "topics": ["<t1>", "<t2>", ...]}}

CONVERSATION TURN:
---
{turn}
---
"""


def extract_summary_topics(text, speaker):
    prompt = SUMMARY_PROMPT.format(turn=text, speaker=speaker)
    result = llm_json(prompt)
    return {
        "summary": result.get("summary", text[:80]),
        "topics": result.get("topics", []),
    }


In [ ]:
# FASE 1 — Extracción (entidades + resumen + tópicos)

def phase1_extract(text, speaker):
    entities = extract_entities(text, speaker=speaker)
    if not any(e["name"] == speaker for e in entities):
        entities.append({"name": speaker, "type": "person"})

    st = extract_summary_topics(text, speaker=speaker)
    return {
        "entities": entities,
        "summary": st["summary"],
        "topics": st["topics"],
    }


## Test de Fase 1



In [ ]:
test_turns = [
    "I live in San Francisco with my partner Sam.",
    "My favorite coffee shop is Sightglass and I go there every morning before work.",
    "I just moved from Seattle to New York for a new job at Stripe as a software engineer.",
    "Yesterday I went to a LGBTQ support group and it was really powerful.",
    "I have two kids, Ava and Noah, and they keep me busy.",
]

for i, turn in enumerate(test_turns, 1):
    print(f"\n{'='*72}\nTURNO {i}: {turn}\n{'='*72}")
    result = phase1_extract(turn, speaker="user")
    print(f"  RESUMEN  : {result['summary']}")
    print(f"  TÓPICOS  : {result['topics']}")
    print(f"  ENTIDADES ({len(result['entities'])}):")
    for e in result['entities']:
        print(f"     - {e['name']:30s} ({e['type']})")



TURNO 1: I live in San Francisco with my partner Sam.
  RESUMEN  : The user lives in San Francisco with their partner Sam.
  TÓPICOS  : ['location', 'relationship']
  ENTIDADES (3):
     - san_francisco                  (location)
     - sam                            (person)
     - user                           (person)

TURNO 2: My favorite coffee shop is Sightglass and I go there every morning before work.
  RESUMEN  : The user's favorite coffee shop is Sightglass, which they visit daily.
  TÓPICOS  : ['coffee', 'shop']
  ENTIDADES (2):
     - sightglass                     (organization)
     - user                           (person)

TURNO 3: I just moved from Seattle to New York for a new job at Stripe as a software engineer.
  RESUMEN  : The user moved from Seattle to New York for a job at Stripe.
  TÓPICOS  : ['move', 'job']
  ENTIDADES (4):
     - seattle                        (location)
     - new_york                       (location)
     - stripe                        

# Fase 2 — Actualización del grafo
1. **Conflict resolution (Mem0g)**: si la entidad ya existe, el LLM decide *ADD / MODIFY / KEEP* sobre sus atributos.
2. **Recálculo de $r(t_i) = \alpha \cdot \text{ant} + \beta \cdot \text{men} + \gamma \cdot \text{ult}$** para todos los turnos.
3. **Poda condicional** si se excede $N_{\max}$.

In [ ]:
# ADD    : el turno aporta info NUEVA -> añadir a attributes
# MODIFY : el turno CONTRADICE o ACTUALIZA info previa -> reemplazar
# KEEP   : el turno no aporta info estructural -> no tocar attributes

ENTITY_CONFLICT_PROMPT = """You are a memory system maintaining structured attributes for named entities.

Given:
  - An entity already in memory with its CURRENT attributes (may be empty)
  - A NEW turn that mentions this entity

Decide ONE action:
  - "ADD"    : new turn reveals NEW facts about this entity -> add to attributes.
  - "MODIFY" : new turn CONTRADICTS or UPDATES existing facts -> replace values.
  - "KEEP"   : no new factual attribute, just re-mention -> keep unchanged.

Attribute values must be CONCISE strings (1-5 words).

Examples:

  Entity: "toby"  | Attributes: {{}}
  Turn: "I adopted Toby last Saturday."
  -> {{"action": "ADD", "attributes": {{"adopted_on": "last Saturday"}}}}

  Entity: "toby"  | Attributes: {{"adopted_on": "last Saturday"}}
  Turn: "Toby is a two-year-old beagle from Refugio Esperanza."
  -> {{"action": "ADD", "attributes": {{"adopted_on": "last Saturday", "age": "2 years", "breed": "beagle", "origin": "Refugio Esperanza"}}}}

  Entity: "toby"  | Attributes: {{"age": "2 years", "breed": "beagle"}}
  Turn: "Actually Toby just turned 3 years old."
  -> {{"action": "MODIFY", "attributes": {{"age": "3 years", "breed": "beagle"}}}}

  Entity: "toby"  | Attributes: {{"breed": "beagle"}}
  Turn: "I took Toby to the beach today."
  -> {{"action": "KEEP", "attributes": {{"breed": "beagle"}}}}

  Entity: "glovo" | Attributes: {{}}
  Turn: "It is at a startup called Glovo and I work as a backend engineer."
  -> {{"action": "ADD", "attributes": {{"type": "startup", "user_role": "backend engineer"}}}}

ENTITY: {entity_name}
CURRENT ATTRIBUTES: {current_attributes}

NEW TURN (speaker={speaker}):
---
{turn_text}
---

Output ONLY this JSON:
{{"action": "ADD" | "MODIFY" | "KEEP", "attributes": {{...COMPLETE new attributes dict...}}}}
"""


def resolve_entity_conflict(entity_name, current_attributes, turn_text, speaker):
    try:
        prompt = ENTITY_CONFLICT_PROMPT.format(
            entity_name=entity_name,
            current_attributes=json.dumps(current_attributes),
            speaker=speaker,
            turn_text=turn_text,
        )
        result = llm_json(prompt)
        action = result.get("action", "KEEP")
        attrs = result.get("attributes", current_attributes)
        if action not in ("ADD", "MODIFY", "KEEP"):
            action = "KEEP"
            attrs = current_attributes
        if not isinstance(attrs, dict):
            attrs = current_attributes
        return {"action": action, "attributes": attrs}
    except Exception as e:
        print(f"[resolve_entity_conflict warning] {entity_name}: {e}")
        return {"action": "KEEP", "attributes": current_attributes}


In [ ]:
import math
import networkx as nx
from datetime import datetime, timezone


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def _build_embed_text(summary, topics):
    topics_str = ", ".join(topics) if topics else ""
    return f"{summary}\nTopics: {topics_str}" if topics_str else summary


class ConvMemoryGraph:

    def __init__(self,
                 alpha=0.3, beta=0.4, gamma=0.3, lam=0.3, n_max=30,
                 exclude_speaker_from_relevance=True,
                 compute_embeddings=True,
                 enable_conflict_resolution=True):
        self.g = nx.MultiDiGraph()
        self.turn_counter = 0
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.lam = lam
        self.n_max = n_max
        self.exclude_speaker_from_relevance = exclude_speaker_from_relevance
        self.compute_embeddings = compute_embeddings #	Calcula embeddings
        self.enable_conflict_resolution = enable_conflict_resolution #Llama al LLM 

    # Inserción de turnos
    def add(self, text, speaker="user", kind="statement"):

        # Fase 1: extraer la info 
        ext = phase1_extract(text, speaker)
        summary = ext["summary"]
        topics = ext["topics"]

        # Embedding del summary + topics (mismo para statement y query)
        if self.compute_embeddings:
            embedding = llm_embed(_build_embed_text(summary, topics))
        else:
            embedding = None

        position = self.turn_counter
        turn_id = f"t{position}"
        self.g.add_node(
            turn_id,
            node_type="turn",
            position=position,
            summary=summary,
            topics=topics,
            role=speaker,
            kind=kind,
            is_query=(kind == "query"),
            embedding=embedding,
            r=1.0,
            created_at=now_iso(),
        )

        # Nodos entidad + aristas MENTIONS
        # Fase 2: para entidades conflict resolution 
        n_conflict_calls = 0
        for e in ext["entities"]:
            name, etype = e["name"], e["type"]
            is_new = name not in self.g.nodes
            is_speaker_ent = (name == speaker)

            if is_new:
                # Nueva entidad -> crear con attributes vacíos
                self.g.add_node(
                    name,
                    node_type="entity",
                    entity_type=etype,
                    attributes={},
                    first_seen=position,
                    last_seen=position,
                    is_speaker=is_speaker_ent,
                    created_at=now_iso(),
                )
            else:
                # Entidad ya existe -> Fase 2 (Mem0g conflict resolver)
                if (self.enable_conflict_resolution
                        and not is_speaker_ent
                        and kind != "query"):
                    current_attrs = self.g.nodes[name].get("attributes", {})
                    resolution = resolve_entity_conflict(
                        entity_name=name,
                        current_attributes=current_attrs,
                        turn_text=text,
                        speaker=speaker,
                    )
                    n_conflict_calls += 1
                    if resolution["action"] != "KEEP":
                        self.g.nodes[name]["attributes"] = resolution["attributes"]
                # Siempre actualizar last_seen
                self.g.nodes[name]["last_seen"] = position

            self.g.add_edge(turn_id, name,
                            edge_type="MENTIONS",
                            created_at=now_iso())

        # Recalcular r(t_i) para todos los turnos
        t_actual = position
        for tid in self._turn_ids():
            self.g.nodes[tid]["r"] = self._compute_r(tid, t_actual)

        # Poda si excedemos n_max
        n_pruned = 0
        if self._n_turns() > self.n_max:
            n_pruned = self._prune_low_relevance_turns()

        self.turn_counter += 1
        return {
            "turn_id": turn_id,
            "position": position,
            "role": speaker,
            "kind": kind,
            "n_entities": len(ext["entities"]),
            "n_conflict_calls": n_conflict_calls,
            "has_embedding": embedding is not None,
            "n_pruned": n_pruned,
        }

    # r(t_i) = α·ant + β·men + γ·ult
    def _compute_r(self, turn_id, t_actual):
        i = self.g.nodes[turn_id]["position"]

        # Antigüedad
        ant = math.exp(-self.lam * (t_actual - i))

        # Menciones: turnos posteriores que comparten alguna entidad
        entities_i = self._entities_of_turn(turn_id)
        sharing_count = 0
        last_mention_j = i

        for tid in self._turn_ids():
            j = self.g.nodes[tid]["position"]
            if j < i:
                continue
            if j == i:
                sharing_count += 1  # auto-mención
                continue
            entities_j = self._entities_of_turn(tid)
            if entities_i & entities_j:
                sharing_count += 1
                last_mention_j = max(last_mention_j, j)

        men = sharing_count / (t_actual - i + 1)
        ult = math.exp(-self.lam * (t_actual - last_mention_j))
        return self.alpha * ant + self.beta * men + self.gamma * ult

    def _turn_ids(self):           # IDs de todos los nodos turno
        return [n for n, d in self.g.nodes(data=True)
                if d.get("node_type") == "turn"]

    def _entity_ids(self):         # IDs de todos los nodos entidad
        return [n for n, d in self.g.nodes(data=True)
                if d.get("node_type") == "entity"]

    def _n_turns(self):            # cuántos turnos hay
        return len(self._turn_ids())

    def _entities_of_turn(self, turn_id, exclude_speaker=None): # qué entidades menciona un turno
        if exclude_speaker is None:
            exclude_speaker = self.exclude_speaker_from_relevance
        out = set()
        for _, v, d in self.g.out_edges(turn_id, data=True):
            if d.get("edge_type") != "MENTIONS":
                continue
            if exclude_speaker and self.g.nodes[v].get("is_speaker"):
                continue
            out.add(v)
        return out

    def _turns_mentioning_entity(self, entity_name, include_queries=False): # qué turnos mencionan una entidad
        if entity_name not in self.g.nodes:
            return []
        turns = []
        for u, _, d in self.g.in_edges(entity_name, data=True):
            if d.get("edge_type") != "MENTIONS":
                continue
            node = self.g.nodes[u]
            if node.get("node_type") != "turn":
                continue
            if not include_queries and node.get("is_query"):
                continue
            turns.append(u)
        return turns

    def _prune_low_relevance_turns(self):
        n_to_remove = self._n_turns() - self.n_max
        if n_to_remove <= 0:
            return 0
        turns_sorted = sorted(self._turn_ids(),
                              key=lambda tid: self.g.nodes[tid]["r"])
        for tid in turns_sorted[:n_to_remove]:
            self.g.remove_node(tid)
        return n_to_remove

    # Visualización
    def show_state(self):
        print(f"\n{'='*72}")
        print(f"ESTADO DEL GRAFO  |  turnos: {self._n_turns()}  "
              f"|  entidades: {len(self._entity_ids())}")
        print('='*72)

        print(f"\nNODOS DE TURNO:")
        for tid in sorted(self._turn_ids(),
                          key=lambda x: self.g.nodes[x]["position"]):
            d = self.g.nodes[tid]
            menciona = sorted(self._entities_of_turn(tid, exclude_speaker=False))
            bar = "█" * int(d["r"] * 20) + "·" * (20 - int(d["r"] * 20))
            kind_tag = " [QUERY]" if d.get("is_query") else ""
            emb_tag = "·emb" if d.get("embedding") else ""
            print(f"  [{tid}] pos={d['position']}  role={d.get('role','?')}{kind_tag}{emb_tag}  "
                  f"r={d['r']:.3f}  {bar}")
            print(f"        resumen:  {d['summary']}")
            print(f"        tópicos:  {d['topics']}")
            print(f"        menciona: {menciona}")
            print()

        print(f"NODOS DE ENTIDAD:")
        for eid in sorted(self._entity_ids()):
            d = self.g.nodes[eid]
            mentioning = sorted(self._turns_mentioning_entity(eid,
                                                              include_queries=True))
            flag = " (speaker)" if d.get("is_speaker") else ""
            attrs = d.get("attributes", {})
            print(f"  [{eid}]{flag}  tipo={d['entity_type']}  "
                  f"first=t{d['first_seen']}, last=t{d['last_seen']}  "
                  f"mencionada por: {mentioning}")
            if attrs:
                print(f"        attributes: {attrs}")

    def show_state_compact(self, last_n=None):
        n_t = self._n_turns()
        n_e = len(self._entity_ids())
        print(f"\n{'─'*72}")
        print(f"GRAFO  |  turnos: {n_t}  |  entidades: {n_e}")
        print('─'*72)

        tids = sorted(self._turn_ids(),
                      key=lambda x: self.g.nodes[x]["position"])
        if last_n is not None and len(tids) > last_n:
            print(f"  ... ({len(tids) - last_n} turnos anteriores omitidos)")
            tids = tids[-last_n:]

        for tid in tids:
            d = self.g.nodes[tid]
            tag = "Q" if d.get("is_query") else "S"
            print(f"  [{tid:>4}] pos={d['position']:>2} {tag} "
                  f"role={d.get('role','?'):<9} r={d['r']:.2f}  "
                  f"{d['summary'][:55]}")

        print(f"\n  Entidades:")
        for eid in sorted(self._entity_ids()):
            d = self.g.nodes[eid]
            mentions = self._turns_mentioning_entity(eid, include_queries=True)
            flag = "*" if d.get("is_speaker") else " "
            attrs = d.get("attributes", {})
            attr_str = ""
            if attrs:
                attr_str = "  " + ", ".join(f"{k}={v}" for k, v in attrs.items())
            print(f"   {flag} {eid:<30s} ({d['entity_type']:<12}) "
                  f"-> {len(mentions)} turnos: {sorted(mentions)}{attr_str}")


# Fase 3 — Recuperación de contexto + Respuesta del LLM


In [ ]:
ANSWER_PROMPT_C1 = """You are a memory assistant answering questions about a user.

The MEMORIES below are summaries of past turns, sorted in REVERSE CHRONOLOGICAL ORDER.

ANSWER RULES:
1. Be CONCISE — output only the answer, no preamble.
2. Match the answer to the question's intent:
   - "Where does X live?" -> a geographic place (city, country, address).
   - "Where does X work?" -> an EMPLOYER (company, organization), NOT a city.
   - "Where did X buy ...?" -> a store or location of purchase.
   - "Who ..." -> a person's name.
   - "How many / how much ..." -> a number or amount.
   - "When ..." -> a date or time.
   - "What sport / hobby / activity does X do?" -> the SPECIFIC activity name
     (e.g. "running", "padel", "cooking") even if the question uses a
     general category word.
   - "Why ..." -> a CAUSE that may be in a different turn than the question's
     entity. Look across all memories for the cause/reason.
3. CONNECT MEMORIES: the answer may require combining facts from
   different turns. E.g. "Why see Dr. Ferrer?" -> look for an injury or
   condition in OTHER turns, not only those that mention Dr. Ferrer.
4. RECENCY: if memories conflict, trust the more recent (higher position).
5. Read carefully: "moved to City for a job at Company" means:
   - "lives in" -> City
   - "works at" -> Company
6. NEVER output an internal identifier in snake_case (e.g. "user_home",
   "speaker_brother"). Those are placeholders. Rephrase in natural language.
7. Only say "I don't know" if the memories truly contain no relevant info.

MEMORIES (MOST RECENT FIRST):
{context}

QUESTION: {query}

ANSWER:"""


def cosine_sim(a, b):
    if not a or not b:
        return 0.0
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    if na == 0 or nb == 0:
        return 0.0
    return dot / (na * nb)


def build_context(mem, turn_ids):
    sorted_tids = sorted(turn_ids,
                         key=lambda t: mem.g.nodes[t]["position"],
                         reverse=True)
    lines = []
    for tid in sorted_tids:
        d = mem.g.nodes[tid]
        lines.append(f"[Turn {d['position']} | role={d.get('role','?')} | r={d['r']:.2f}]")
        lines.append(f"Summary: {d['summary']}")
        if d.get("topics"):
            lines.append(f"Topics: {', '.join(d['topics'])}")
        mentions = sorted(mem._entities_of_turn(tid, exclude_speaker=False))
        if mentions:
            ents = ", ".join(
                f"{m} ({mem.g.nodes[m].get('entity_type','?')})" for m in mentions
            )
            lines.append(f"Mentions: {ents}")
        lines.append("")
    return "\n".join(lines).strip()


# ----------------------------------------------------------------------
# Recuperación: ranking híbrido sin filtro estricto
# ----------------------------------------------------------------------
def retrieve_relevant_turns(mem, query, speaker="user",
                            k=5,
                            w_sim=0.6, w_r=0.2, w_entity=0.2):

    # 1) Insertar la pregunta como turno kind='query' (con embedding)
    r_info = mem.add(query, speaker=speaker, kind="query")
    q_id, q_pos = r_info["turn_id"], r_info["position"]
    q_emb = mem.g.nodes[q_id].get("embedding")
    q_entities = mem._entities_of_turn(q_id, exclude_speaker=True)

    # 2) Candidatos = TODOS los statements anteriores (sin filtros estrictos)
    candidates = [
        tid for tid in mem._turn_ids()
        if tid != q_id
        and mem.g.nodes[tid]["position"] < q_pos
        and not mem.g.nodes[tid].get("is_query")
    ]

    # 3) Calcular score híbrido por candidato
    breakdown = {}
    for tid in candidates:
        node = mem.g.nodes[tid]
        # 3a) Similitud semántica (embedding de la pregunta vs del turno)
        sim = max(0.0, cosine_sim(q_emb, node.get("embedding"))) if q_emb else 0.0
        # 3b) Relevancia estructural r(t_i) = α·ant + β·men + γ·ult
        r_val = node.get("r", 0.0)
        # 3c) Bonus por compartir entidad con la pregunta
        t_entities = mem._entities_of_turn(tid, exclude_speaker=True)
        entity_match = 1.0 if (q_entities & t_entities) else 0.0
        # 3d) Score combinado
        score = w_sim * sim + w_r * r_val + w_entity * entity_match
        breakdown[tid] = (score, sim, r_val, entity_match)

    # 4) Top-k por score
    ranked = sorted(candidates, key=lambda t: breakdown[t][0], reverse=True)
    return q_id, ranked[:k], breakdown


def answer(mem, query, speaker="user",
           k=5,
           w_sim=0.6, w_r=0.2, w_entity=0.2,
           verbose=False):
    q_id, top_turns, breakdown = retrieve_relevant_turns(
        mem, query, speaker=speaker,
        k=k, w_sim=w_sim, w_r=w_r, w_entity=w_entity,
    )
    if not top_turns:
        return "I don't know — no memories available yet."

    context = build_context(mem, top_turns)
    prompt = ANSWER_PROMPT_C1.format(context=context, query=query)

    if verbose:
        d = mem.g.nodes[q_id]
        print("=== TURNO DE LA PREGUNTA ===")
        print(f"  {q_id}  pos={d['position']}  role={d.get('role')}  kind={d.get('kind')}")
        print(f"  resumen: {d['summary']}")
        print(f"  entidades: {sorted(mem._entities_of_turn(q_id, exclude_speaker=False))}")
        print(f"  pesos: w_sim={w_sim}, w_r={w_r}, w_entity={w_entity}, k={k}")

        print(f"\n=== TOP-{len(top_turns)} (score = {w_sim}·sim + {w_r}·r + {w_entity}·ent) ===")
        for tid in top_turns:
            score, sim, r_val, ent = breakdown[tid]
            summary = mem.g.nodes[tid]["summary"]
            ent_tag = "✓" if ent > 0 else " "
            print(f"  {tid}  score={score:.3f}  (sim={sim:.3f}, r={r_val:.3f}, ent={ent_tag})  {summary[:55]}")

        print("\n=== CONTEXTO ENVIADO AL LLM ===")
        print(context)
        print(f"\nQUESTION: {query}")
        print("=" * 50)

    return llm_text(prompt)


In [ ]:
CONVERSATION_PATH = "test_conversation.txt"


def load_conversation(path):

    turns = []
    with open(path, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("USER:"):
                turns.append(("user", line[len("USER:"):].strip()))
            elif line.startswith("ASSISTANT:"):
                turns.append(("assistant", line[len("ASSISTANT:"):].strip()))
    return turns



turns = load_conversation(CONVERSATION_PATH)
print(f"Dataset cargado: {len(turns)} turnos\n")

# Crear grafo vacío
mem_txt = ConvMemoryGraph(alpha=0.3, beta=0.4, gamma=0.3, lam=0.3, n_max=50)

for i, (role, text) in enumerate(turns):
    print(f"{'─'*72}")
    print(f"TURNO {i}  [role={role}]  {text}")

    ext = phase1_extract(text, speaker=role)
    ents = ", ".join(f"{e['name']}({e['type']})" for e in ext["entities"])
    print(f"  RESUMEN  : {ext['summary']}")
    print(f"  TÓPICOS  : {ext['topics']}")
    print(f"  ENTIDADES: {ents}")

    r = mem_txt.add(text, speaker=role)
    print(f"  -> {r['turn_id']} (pos={r['position']}, {r['n_entities']} entidades)\n")

print("\n" + "#"*72)
print("# GRAFO RESULTANTE")
print("#"*72)
mem_txt.show_state()


Dataset cargado: 26 turnos

────────────────────────────────────────────────────────────────────────
TURNO 0  [role=user]  I just moved from Valencia to Barcelona last week for a new job.
  RESUMEN  : The user moved from Valencia to Barcelona last week for a new job.
  TÓPICOS  : ['moving', 'job']
  ENTIDADES: valencia(location), barcelona(location), user(person)
  -> t0 (pos=0, 3 entidades)

────────────────────────────────────────────────────────────────────────
TURNO 1  [role=assistant]  Congratulations on the move to Barcelona. How is the new job going so far?
  RESUMEN  : The assistant discussed a recent move to Barcelona and a new job.
  TÓPICOS  : ['move', 'new job']
  ENTIDADES: barcelona(location), assistant(person)
  -> t1 (pos=1, 2 entidades)

────────────────────────────────────────────────────────────────────────
TURNO 2  [role=user]  It is at a startup called Glovo and I work as a backend engineer.
  RESUMEN  : The user works as a backend engineer at Glovo.
  TÓPICOS  : [

In [ ]:
preguntas = [
    "What sport do I do regularly?",
    "What day do I have cooking class?",
    "What is my role at work?",
    "How many kilometers do I usually run?",
    "Where do I work?",
    "Who is my manager at Glovo?",
    "Why do I need to see Dr. Ferrer?",
    "In what neighborhood is La Tavola?",
    "What's the connection between Marco and La Tavola?",
    "In what city is Clinica Diagonal?",
    "What is Toby's age?",
    "What breed is Toby?",
    "When did I adopt my dog?",
    "Where does my dog Toby come from?",
]

n_turns_antes = mem_txt._n_turns()
print(f"Turnos en el grafo antes del chat: {n_turns_antes}\n")

for i, q in enumerate(preguntas, 1):
    print("\n" + "█"*72)
    print(f"# PREGUNTA {i}/{len(preguntas)}")
    print("█"*72)

    ans = answer(mem_txt, q, k=5, verbose=True)
    print(f"\n>>> RESPUESTA DEL LLM: {ans}")

    r = mem_txt.add(ans, speaker="assistant", kind="statement")
    print(f"    [+] respuesta añadida al grafo como {r['turn_id']} "
          f"(pos={r['position']}, role=assistant, kind=statement)")


Turnos en el grafo antes del chat: 26


████████████████████████████████████████████████████████████████████████
# PREGUNTA 1/14
████████████████████████████████████████████████████████████████████████
=== TURNO DE LA PREGUNTA ===
  t26  pos=26  role=user  kind=query
  resumen: The user does a sport regularly.
  entidades: ['user']
  pesos: w_sim=0.6, w_r=0.2, w_entity=0.2, k=5

=== TOP-5 (score = 0.6·sim + 0.2·r + 0.2·ent) ===
  t23  score=0.477  (sim=0.580, r=0.644, ent= )  The user mentioned Sofia helping and Toby being in good
  t24  score=0.468  (sim=0.562, r=0.654, ent= )  The user plans to take Toby to the beach in Barceloneta
  t25  score=0.467  (sim=0.564, r=0.644, ent= )  The user mentioned Barceloneta as a suitable location f
  t18  score=0.466  (sim=0.590, r=0.561, ent= )  The user adopted a dog named Toby on last Saturday.
  t20  score=0.436  (sim=0.541, r=0.558, ent= )  The user mentioned Toby, a two-year-old beagle from the

=== CONTEXTO ENVIADO AL LLM ===
[Turn 25 | rol

In [ ]:
n_turns_despues = mem_txt._n_turns()
print("\n\n" + "="*72)
print(f"CHAT TERMINADO")
print(f"  Turnos antes  : {n_turns_antes}")
print(f"  Turnos después: {n_turns_despues}  (+{n_turns_despues - n_turns_antes})")
print(f"  Esperado      : {len(preguntas)} preguntas + {len(preguntas)} respuestas "
      f"= {2 * len(preguntas)} turnos nuevos")
print("="*72)

print("\n### ESTADO FINAL DEL GRAFO (vista compacta, últimos 40 turnos)")
mem_txt.show_state_compact(last_n=40)



CHAT TERMINADO
  Turnos antes  : 26
  Turnos después: 50  (+24)
  Esperado      : 14 preguntas + 14 respuestas = 28 turnos nuevos

### ESTADO FINAL DEL GRAFO (vista compacta, últimos 40 turnos)

────────────────────────────────────────────────────────────────────────
GRAFO  |  turnos: 50  |  entidades: 28
────────────────────────────────────────────────────────────────────────
  ... (10 turnos anteriores omitidos)
  [ t14] pos=14 S role=user      r=0.01  The user usually runs 8 kilometers, but had to stop due
  [ t15] pos=15 S role=assistant r=0.01  The assistant asked if the user had seen a doctor for t
  [ t16] pos=16 S role=user      r=0.06  The user has an appointment with Dr. Ferrer at Clinica 
  [ t17] pos=17 S role=assistant r=0.02  The assistant mentioned Dr. Ferrer's potential assistan
  [ t18] pos=18 S role=user      r=0.34  The user adopted a dog named Toby on last Saturday.
  [ t19] pos=19 S role=assistant r=0.33  The assistant discussed a pet named Toby.
  [ t20] pos=20 